In [1]:
import numpy as np
import pandas as pd
from rapidfuzz import process, fuzz
from sklearn.metrics.pairwise import cosine_similarity
import os
import requests
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

In [2]:
df=pd.read_csv(r"C:\Users\Asus\Downloads\Jupiter Py Pro\IMDB Project\ML- Layer\Step 0- Cleaning\Clean_Data.csv")
embeddings=np.load("movie_embeddings.npy") 


In [3]:
load_dotenv()
OMDB_API_KEY = os.getenv("OMDB_API_KEY")
model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
def fetch_omdb(title):
    try:
        resp = requests.get("http://www.omdbapi.com/", params={
            "t": title, "type": "movie", "plot": "full", "apikey": OMDB_API_KEY
        }, timeout=10)
        data = resp.json()
    except requests.exceptions.RequestException as e:
        print(f"Network error contacting OMDb: {e}")
        return None

    if data.get("Response") == "True" and data.get("Plot", "N/A") != "N/A":
        return {
            "name": data["Title"],
            "year": data.get("Year", "")[:4],
            "lead_actor": data.get("Actors", "").split(",")[0].strip(),
            "director": data.get("Director", ""),
            "imdb_rating": data.get("imdbRating", ""),
            "genre": data.get("Genre", ""),
            "plot": data["Plot"],
        }
    return None
def fetch_wikipedia(title):
    resp = requests.get(f"https://en.wikipedia.org/api/rest_v1/page/summary/{title}", timeout=10)
    if resp.status_code == 200:
        data = resp.json()
        extract = data.get("extract", "")
        if extract and len(extract.split()) > 15:  # too short = probably not useful as a plot
            return {
                "name": title, "year": "", "lead_actor": "", "director": "",
                "imdb_rating": "", "genre": "", "plot": extract,
            }
    return None
def ask_user_for_synopsis(title):
    print(f"Couldn't find '{title}' anywhere automatically.")
    print("Paste the official synopsis if you have one (e.g. from Wikipedia/IMDb),")
    print("or just describe the plot in a couple sentences:")
    plot = input("> ").strip()
    if len(plot.split()) < 5:
        return None
    return {
        "name": title, "year": "", "lead_actor": "", "director": "",
        "imdb_rating": "", "genre": "", "plot": plot,
    }

def add_to_dataset(movie_dict):
    global df, embeddings
    new_vec = model.encode([movie_dict["plot"]])
    
    new_embeddings = np.vstack([embeddings, new_vec])
    new_df = pd.concat([df, pd.DataFrame([movie_dict])], ignore_index=True)

    try:
        new_df.to_csv("Clean_Data.csv", index=False)
        np.save("movie_embeddings.npy", new_embeddings)
        header_needed = not os.path.exists("live_additions.csv")
        pd.DataFrame([movie_dict]).to_csv("live_additions.csv", mode="a", index=False, header=header_needed)
    except PermissionError as e:
        print(f"Could not save (file may be open elsewhere): {e}")
        print("Not updating in-memory data either, to avoid drift. Close the file and try again.")
        return None  # neither df, embeddings, nor disk changed — everything stays consistent

    # Only commit to the in-memory objects once BOTH saves succeeded
    df = new_df
    embeddings = new_embeddings
    return len(df) - 1


In [5]:
import re

def normalize(title):
    title = str(title).lower().strip()
    title=re.sub(r'[:\---]','',title)
    roman_to_num={"i":"1","ii":"2","iii":"3","iv":"4","v":"5"}
    def convert_part(match):
        num=match.group(1).lower()
        return f'part{roman_to_num.get(num, num)}'
        title=re.sub(r'\bpart\s+(i{1,3}|iv|v|\d+)\b)', convert_part,title)
        title=re.sub(r'\s+','',title).strip()
        
    return title

def find_local(title, threshold=85):
    query = normalize(title)
    candidates = df["name"].apply(normalize)

    # If user supplied a year, it MUST match
    year_match = re.search(r'\b(19|20)\d{2}\b', title)

    # Also understand "Heat of 95" / "Heat 95"
    if not year_match:
        short_year_match = re.search(r'\b(?:of\s+)?(\d{2})\b', title)

        if short_year_match:
            short_year = int(short_year_match.group(1))

            if short_year <= 30:
                year = 2000 + short_year
            else:
                year = 1900 + short_year

            title_without_year = re.sub(
                r'\b(?:of\s+)?\d{2}\b',
                '',
                title,
                count=1
            )

            query = normalize(title_without_year)

            candidates = candidates[df["year"] == year]

            # Requested year doesn't exist → DON'T fuzzy-match another year
            if len(candidates) == 0:
                return None

    else:
        year = int(year_match.group())

        title_without_year = re.sub(
            r'\b(19|20)\d{2}\b',
            '',
            title
        )

        query = normalize(title_without_year)

        candidates = candidates[df["year"] == year]

        # Requested year doesn't exist → DON'T fuzzy-match another year
        if len(candidates) == 0:
            return None

    # 1. Exact match FIRST
    exact_matches = candidates[candidates == query]

    if len(exact_matches) > 0:
        return exact_matches.index[0]

    # 2. If user explicitly gave a Part number,
    #    NEVER use fuzzy matching
    if re.search(r'\bpart\b', query):
        return None

    # 3. Fuzzy matching only for normal titles / typos
    match = process.extractOne(
        query,
        candidates,
        scorer=fuzz.token_sort_ratio
    )

    if match is None or match[1] < threshold:
        return None

    return match[2]

    return match[2]
def recommend(idx, top_n=5):
    """Given a row index, find its top_n most similar movies by plot embedding."""
    query_vec = embeddings[idx].reshape(1, -1)
    sims = cosine_similarity(query_vec, embeddings)[0]
    results=df.copy()
    results['similarity']=sims
    results['bonus']=0.0
    results.loc[results['director']==df.iloc[idx]['director'],'bonus']+=0.05
    results.loc[results['lead_actor']==df.iloc[idx]['lead_actor'],'bonus']+=0.03
    if "genre" in df.columns:
        query_genres = set(str(df.iloc[idx]['genre']).lower().split(", "))
        for movie_idx in results.index:
                movie_genres = set(str(results.loc[movie_idx]['genre']).lower().split(", "))
                shared_genres = query_genres & movie_genres
                genre_bonus = min(0.02 * len(shared_genres), 0.04)
                results.loc[movie_idx, "bonus"] += genre_bonus
           
    results['final_score']=results['similarity']+results['bonus']
    results=results.drop(index=idx)
    results=results.sort_values(by='final_score',ascending=False).head(top_n)
    return results[['name','year','lead_actor','director','imdb_rating','plot','similarity','bonus','final_score']]
    
def get_recommendations(title, top_n=5):
    idx = find_local(title)

    if idx is None:
        print(f"'{title}' not in local dataset, checking OMDb...")
        movie = fetch_omdb(title)

        if movie is None:
            print("Not on OMDb either, trying Wikipedia...")
            movie = fetch_wikipedia(title)

        if movie is None:
            movie = ask_user_for_synopsis(title)

        if movie is None:
            print("Not enough info to recommend anything.")
            return None

        idx = add_to_dataset(movie)
        print(f"Added '{movie['name']}' to the dataset for future searches.\n")

    print(f"Because you liked: {df.iloc[idx]['name']} ({df.iloc[idx]['year']})\n")
    results = recommend(idx, top_n)

    for _, row in results.iterrows():
        snippet = row["plot"][:100] + "..." if len(row["plot"]) > 100 else row["plot"]
        print(f"{row['name']} ({row['year']}) — dir. {row['director']}, starring {row['lead_actor']} — IMDb {row['imdb_rating']}")
        print(f"   {snippet}")
        print(f"   match score: {row['similarity']:.2f}\n")

    return results


In [11]:
get_recommendations("Exorcist")

'Exorcist' not in local dataset, checking OMDb...
Added 'The Exorcist' to the dataset for future searches.

Because you liked: The Exorcist (1973)

The Haunting of Morella (1990) — dir. Jim Wynorski, starring Nicole Eggert — IMDb 4.4
   A witch is put to death in Colonial America, leaving her husband and infant daughter behind. Sevente...
   match score: 0.55

The Grudge (2004) — dir. Takashi Shimizu, starring Sarah Michelle Gellar — IMDb 5.9
   Karen Davis, an American Nurse, moves to Tokyo and encounters a supernatural spirit who is vengeful ...
   match score: 0.53

The Heart Is Deceitful Above All Things (2004) — dir. Asia Argento, starring Asia Argento — IMDb 6.2
   The dysfunctional twenty-three-year-old Sarah takes her six-year-old natural son Jeremiah from the h...
   match score: 0.55

The Rapture (1991) — dir. Michael Tolkin, starring Mimi Rogers — IMDb 6.3
   This is the story of a young woman (who lives in Los Angeles) with a very boring job. At night howev...
   match scor

,name,year,lead_actor,director,imdb_rating,plot,similarity,bonus,final_score
1814,The Haunting of Morella,1990,Nicole Eggert,Jim Wynorski,4.4,"A witch is put to death in Colonial America, l...",0.551585,0.02,0.571585
4460,The Grudge,2004,Sarah Michelle Gellar,Takashi Shimizu,5.9,"Karen Davis, an American Nurse, moves to Tokyo...",0.533277,0.02,0.553277
4475,The Heart Is Deceitful Above All Things,2004,Asia Argento,Asia Argento,6.2,The dysfunctional twenty-three-year-old Sarah ...,0.551074,0.00,0.551074
1904,The Rapture,1991,Mimi Rogers,Michael Tolkin,6.3,This is the story of a young woman (who lives ...,0.548456,0.00,0.548456
3480,Stigmata,1999,Patricia Arquette,Rupert Wainwright,6.2,A priest from the Vatican is sent to Sao Paulo...,0.525652,0.02,0.545652


In [7]:
g1 = set(df.loc[df["name"] == "Goodfellas", "genre"].iloc[0].lower().split(", "))
g2 = set(df.loc[df["name"] == "Casino", "genre"].iloc[0].lower().split(", "))

print("Goodfellas:", g1)
print("Casino:", g2)
print("Shared:", g1 & g2)
print("Genre bonus:", min(0.02 * len(g1 & g2), 0.04))

Goodfellas: {'biography', 'drama', 'crime'}
Casino: {'drama', 'crime'}
Shared: {'drama', 'crime'}
Genre bonus: 0.04
